In [ ]:
!gcloud auth application-default login

In [ ]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()


session = GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 15:12:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
path_to_release_folder = "gs://open-targets-data-releases/25.06/"


si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study/")
sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set/")

sl_eff = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect"
)

l2g_full = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet"
)

In [ ]:
v1 = session.spark.read.csv(
    "./data/combined_evidence_for_constrain_plot.csv", header=True, inferSchema=True
)

In [10]:
v1.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [11]:
v1.groupBy("source").count().show()

+--------------------+-----+
|              source|count|
+--------------------+-----+
|        all_diseases| 8285|
|          hc_paralog| 4173|
|    distant_ortholog|  830|
|       cancer_ChEMBL|  649|
|         gene_burden|  557|
|                omim| 4182|
|           inComplex| 3530|
|  cancer_driver_gene|  368|
|      withdrawn_drug|  166|
|   non_cancer_ChEMBL|  958|
|       selectiveGene|  930|
|      essential_gene| 1489|
|           gwas_eQTL| 3945|
|       gwas_with_pav| 1574|
|       liable_target|  214|
|            orphanet| 3814|
|              ChEMBL| 1137|
|          dd_related|  861|
|        pharmacogene|  538|
|trial_safety_concern|  440|
+--------------------+-----+
only showing top 20 rows



# FUSSIL


In [132]:
target = session.spark.read.parquet(
    "gs://open-targets-pipeline-runs/il/25.09-testrun-1/output/target"
)

In [133]:
# Create a list of valid chromosomes
valid_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]


In [134]:
# target=session.spark.read.parquet("gs://open-targets-data-releases/25.06/output/target")
target = (
    target.filter(
        f.col("genomicLocation").getField("chromosome").isin(valid_chromosomes)
    )
    .filter(f.col("biotype") == "protein_coding")
    .cache()
)
target.count()

25/11/20 16:39:54 WARN CacheManager: Asked to cache already cached data.


20083

In [135]:
target.printSchema()

root
 |-- id: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull = tru

In [130]:
target.show(2, truncate=False)

+---------------+--------------+--------------+--------------------------------------------------------------------+---------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------+----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [136]:
mouse_syn = (
    target.select("id", "homologues")
    .select("id", f.explode("homologues").alias("syn"))
    .filter(f.lower(f.col("syn.speciesName")) == "mouse")  # case-insensitive match
    .select("id", f.col("syn.targetGeneId").alias("mouse_symbol"))
    .na.drop(subset=["mouse_symbol"])  # drop null labels if any
    .distinct()  # unique pairs
)

In [137]:
mouse_syn.show()

+---------------+------------------+
|             id|      mouse_symbol|
+---------------+------------------+
|ENSG00000128039|ENSMUSG00000029233|
|ENSG00000134758|ENSMUSG00000024317|
|ENSG00000139351|ENSMUSG00000096468|
|ENSG00000141540|ENSMUSG00000034714|
|ENSG00000149506|ENSMUSG00000024734|
|ENSG00000171962|ENSMUSG00000056598|
|ENSG00000178882|ENSMUSG00000037962|
|ENSG00000179119|ENSMUSG00000049516|
|ENSG00000180694|ENSMUSG00000043252|
|ENSG00000184697|ENSMUSG00000023906|
|ENSG00000204481|ENSMUSG00000066030|
|ENSG00000205133|ENSMUSG00000055963|
|ENSG00000205330|ENSMUSG00000095696|
|ENSG00000273173|ENSMUSG00000102627|
|ENSG00000274070|ENSMUSG00000015944|
|ENSG00000100416|ENSMUSG00000022386|
|ENSG00000124181|ENSMUSG00000016933|
|ENSG00000128699|ENSMUSG00000026097|
|ENSG00000140153|ENSMUSG00000037957|
|ENSG00000158234|ENSMUSG00000032463|
+---------------+------------------+
only showing top 20 rows



In [138]:
mouse_syn.select("id").distinct().count()

17678

In [139]:
mouse_syn.select("mouse_symbol").distinct().count()

18616

In [140]:
mouse_syn.count()

22184

In [141]:
mouse_syn.show(1)

+---------------+------------------+
|             id|      mouse_symbol|
+---------------+------------------+
|ENSG00000128039|ENSMUSG00000029233|
+---------------+------------------+
only showing top 1 row



## Fussil data


In [142]:
fussil = session.spark.read.csv(
    "/Users/yt4/Projects/Gentropy-manuscript/data/SourceDataFile1_FUSIL_bins.txt",
    header=True,
    inferSchema=True,
    sep="\t",
)

In [143]:
mapping = session.spark.read.csv(
    "/Users/yt4/Projects/Gentropy-manuscript/data/MGIBatchReport_20251120_112357.txt",
    header=True,
    inferSchema=True,
    sep="\t",
)

In [144]:
mapping.show(1)

+-----------+----------+------------------+------+--------------------+-------------------+------------------+
|      Input|Input Type|MGI Gene/Marker ID|Symbol|                Name|       Feature Type|        Ensembl ID|
+-----------+----------+------------------+------+--------------------+-------------------+------------------+
|MGI:1098434|       MGI|       MGI:1098434|  Rgs5|regulator of G-pr...|protein coding gene|ENSMUSG00000026678|
+-----------+----------+------------------+------+--------------------+-------------------+------------------+
only showing top 1 row



In [145]:
mapping.count()

4668

In [146]:
fussil.show(1)

+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+
|   HGNC_ID|     MGI_ID|IMPC_adult_viability|IMPC_embryo_viability| Avana_mean|Avana_mean_045|IMPC_Viability|adHOM_top_level_mp_name|adHEMI_top_level_mp_name|Viable_Phenotypes_updated|FUSIL_outliers|FUSIL|
+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+
|HGNC:10001|MGI:1098434|              Viable|    InsuffData/NoData|0.020186795| Non-essential|        Viable|                      -|                       -|       Viable_nophenotype|            VN|   VN|
+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+------------------

In [147]:
fussil.groupBy("FUSIL").count().show()

+-----+-----+
|FUSIL|count|
+-----+-----+
|   VP| 1867|
|   SV|  421|
|   CL|  413|
|   DL|  764|
|    -|  881|
|   VN|  318|
+-----+-----+



In [148]:
fussil.count()

4664

In [149]:
mapping = mapping.select("input", "Ensembl ID").withColumnRenamed("input", "MGI_ID")

In [150]:
fussil = fussil.join(mapping, on="MGI_ID", how="left")

In [151]:
fussil.count()

4668

In [152]:
fussil.show(1)

+-----------+----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+------------------+
|     MGI_ID|   HGNC_ID|IMPC_adult_viability|IMPC_embryo_viability| Avana_mean|Avana_mean_045|IMPC_Viability|adHOM_top_level_mp_name|adHEMI_top_level_mp_name|Viable_Phenotypes_updated|FUSIL_outliers|FUSIL|        Ensembl ID|
+-----------+----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+------------------+
|MGI:1098434|HGNC:10001|              Viable|    InsuffData/NoData|0.020186795| Non-essential|        Viable|                      -|                       -|       Viable_nophenotype|            VN|   VN|ENSMUSG00000026678|
+-----------+----------+--------------------+---------------------+-----------+--------------+------

In [153]:
fussil.printSchema()

root
 |-- MGI_ID: string (nullable = true)
 |-- HGNC_ID: string (nullable = true)
 |-- IMPC_adult_viability: string (nullable = true)
 |-- IMPC_embryo_viability: string (nullable = true)
 |-- Avana_mean: string (nullable = true)
 |-- Avana_mean_045: string (nullable = true)
 |-- IMPC_Viability: string (nullable = true)
 |-- adHOM_top_level_mp_name: string (nullable = true)
 |-- adHEMI_top_level_mp_name: string (nullable = true)
 |-- Viable_Phenotypes_updated: string (nullable = true)
 |-- FUSIL_outliers: string (nullable = true)
 |-- FUSIL: string (nullable = true)
 |-- Ensembl ID: string (nullable = true)



In [154]:
fussil = fussil.select("HGNC_ID", "FUSIL", "Ensembl ID").filter(
    ~(f.col("FUSIL") == "-")
)

In [155]:
fussil.groupBy("FUSIL").count().show()

+-----+-----+
|FUSIL|count|
+-----+-----+
|   VP| 1869|
|   SV|  421|
|   CL|  413|
|   DL|  766|
|   VN|  318|
+-----+-----+



In [156]:
fussil.count()

3787

In [157]:
fussil.select("Ensembl ID").distinct().count()

3787

In [158]:
# fussil = fussil.withColumn(
#    "hgnc_id_small",
#    f.lower(
#        f.regexp_replace(f.col("HGNC_ID").cast("string"), r"(?i)^hgnc:", "")
#    )
# )

In [ ]:
fussil = fussil.withColumnRenamed("Ensembl ID", "mouse_symbol")

In [161]:
fussil.show()

+----------+-----+------------------+
|   HGNC_ID|FUSIL|      mouse_symbol|
+----------+-----+------------------+
|HGNC:10001|   VN|ENSMUSG00000026678|
|HGNC:10007|   DL|ENSMUSG00000025735|
|HGNC:10009|   VP|ENSMUSG00000028825|
|HGNC:10021|   VP|ENSMUSG00000022221|
|HGNC:10024|   VN|ENSMUSG00000039194|
| HGNC:1004|   DL|ENSMUSG00000029438|
|HGNC:10044|   VP|ENSMUSG00000035896|
|HGNC:10057|   VP|ENSMUSG00000036503|
|HGNC:10062|   CL|ENSMUSG00000028309|
|HGNC:10065|   VP|ENSMUSG00000045409|
|HGNC:10075|   CL|ENSMUSG00000009535|
| HGNC:1020|   CL|ENSMUSG00000026172|
|HGNC:10251|   SV|ENSMUSG00000024290|
|HGNC:10258|   SV|ENSMUSG00000032238|
|HGNC:10289|   CL|ENSMUSG00000000751|
|HGNC:10297|   DL|ENSMUSG00000053604|
|  HGNC:103|   DL|ENSMUSG00000064105|
| HGNC:1034|   DL|ENSMUSG00000035086|
| HGNC:1037|   VP|ENSMUSG00000090231|
|HGNC:10378|   CL|ENSMUSG00000039640|
+----------+-----+------------------+
only showing top 20 rows



In [163]:
fussil.count()

3787

In [ ]:
fussil_j = fussil.join(mouse_syn, on="mouse_symbol", how="inner")

In [166]:
fussil_j.count()

3803

In [168]:
fussil_j.show(1)

+------------------+----------+-----+---------------+
|      mouse_symbol|   HGNC_ID|FUSIL|             id|
+------------------+----------+-----+---------------+
|ENSMUSG00000026678|HGNC:10001|   VN|ENSG00000143248|
+------------------+----------+-----+---------------+
only showing top 1 row



In [ ]:
fussil_j = fussil_j.select("FUSIL", "id").distinct()
fussil_j.count()

3802

In [170]:
fussil_j.show()

+-----+---------------+
|FUSIL|             id|
+-----+---------------+
|   VP|ENSG00000144290|
|   VP|ENSG00000149635|
|   CL|ENSG00000117360|
|   DL|ENSG00000115289|
|   VP|ENSG00000124762|
|   DL|ENSG00000076043|
|   VN|ENSG00000204130|
|   VP|ENSG00000120675|
|   VP|ENSG00000204020|
|   VP|ENSG00000163472|
|   VP|ENSG00000247077|
|   VP|ENSG00000165660|
|   VP|ENSG00000131242|
|   VP|ENSG00000120341|
|   VP|ENSG00000136478|
|   VP|ENSG00000243364|
|   VP|ENSG00000131203|
|   VP|ENSG00000171747|
|   DL|ENSG00000241837|
|   CL|ENSG00000196235|
+-----+---------------+
only showing top 20 rows



In [ ]:
fussil_j_prepared = (
    fussil_j.withColumn("source", f.concat(f.lit("fusil_"), f.col("FUSIL")))
    .withColumnRenamed("id", "targetId")
    .select("source", "targetId")
    .distinct()
)
fussil_j_prepared.show(5)


+--------+---------------+
|  source|       targetId|
+--------+---------------+
|fusil_VP|ENSG00000171206|
|fusil_CL|ENSG00000104671|
|fusil_VP|ENSG00000179431|
|fusil_VP|ENSG00000114737|
|fusil_CL|ENSG00000137812|
+--------+---------------+
only showing top 5 rows



# Constrain stratas


In [174]:
target.show(1)

+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+----------------+--------------------+----------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------+---------+
|             id|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes| approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|targetClass| obsoleteSymbols|       obsoleteNames|constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|          homologues|        tractability|safetyLiabilities|pathways|      tss

In [ ]:
target_c = target.select("id", "biotype", "constraint").withColumnRenamed(
    "id", "targetId"
)
target_c.show(5, truncate=False)

+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|targetId       |biotype       |constraint                                                                                                                                                                                               |
+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ENSG00000000003|protein_coding|NULL                                                                                                                                                                                                     |
|ENSG00000000005|protein_coding|NULL                        

In [176]:
target_c.count()

20083

In [ ]:
from pyspark.sql.functions import col, expr, when, size, element_at

target_with_constraints = (
    target_c.withColumn(
        "syn_constr", expr("filter(constraint, x -> x.constraintType = 'syn')[0].score")
    )
    .withColumn(
        "mis_constr", expr("filter(constraint, x -> x.constraintType = 'mis')[0].score")
    )
    .withColumn(
        "lof_constr",
        expr("filter(constraint, x -> x.constraintType = 'lof')[0].oeUpper"),
    )
)

In [178]:
target_with_constraints.show(1)

+---------------+--------------+----------+----------+----------+----------+
|       targetId|       biotype|constraint|syn_constr|mis_constr|lof_constr|
+---------------+--------------+----------+----------+----------+----------+
|ENSG00000000003|protein_coding|      NULL|      NULL|      NULL|      NULL|
+---------------+--------------+----------+----------+----------+----------+
only showing top 1 row



In [ ]:
target_with_constraints = target_with_constraints.withColumn(
    "lof_constr_corrected", -f.col("lof_constr")
)

In [180]:
target_with_constraints.show(1)

+---------------+--------------+----------+----------+----------+----------+--------------------+
|       targetId|       biotype|constraint|syn_constr|mis_constr|lof_constr|lof_constr_corrected|
+---------------+--------------+----------+----------+----------+----------+--------------------+
|ENSG00000000003|protein_coding|      NULL|      NULL|      NULL|      NULL|                NULL|
+---------------+--------------+----------+----------+----------+----------+--------------------+
only showing top 1 row



In [ ]:
trq = target_with_constraints.dropna(subset=["lof_constr_corrected"])

# 2) compute quartile cutoffs and add quartile column (Q1..Q4)
q1, q2, q3 = trq.approxQuantile("lof_constr_corrected", [0.25, 0.5, 0.75], 0.0)

from pyspark.sql import functions as f

trq = trq.withColumn(
    "source",
    f.when(f.col("lof_constr_corrected") <= f.lit(q1), "lof_constr_Q1")
    .when(
        (f.col("lof_constr_corrected") > f.lit(q1))
        & (f.col("lof_constr_corrected") <= f.lit(q2)),
        "lof_constr_Q2",
    )
    .when(
        (f.col("lof_constr_corrected") > f.lit(q2))
        & (f.col("lof_constr_corrected") <= f.lit(q3)),
        "lof_constr_Q3",
    )
    .otherwise("lof_constr_Q4"),
).select("targetId", "source")

# quick check
trq.groupBy("source").count().show()

+-------------+-----+
|       source|count|
+-------------+-----+
|lof_constr_Q4| 4520|
|lof_constr_Q2| 4532|
|lof_constr_Q3| 4519|
|lof_constr_Q1| 4526|
+-------------+-----+



In [185]:
trq.show(1)

+---------------+-------------+
|       targetId|       source|
+---------------+-------------+
|ENSG00000001084|lof_constr_Q4|
+---------------+-------------+
only showing top 1 row



# Combine all together


In [186]:
v1.count()

60511

In [187]:
v1.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [ ]:
v2 = v1.unionByName(trq).unionByName(fussil_j_prepared)

In [189]:
v2.count()

82410

In [191]:
v2.groupBy("source").count().count()

32

In [192]:
v1.groupBy("source").count().count()

23

In [195]:
v2.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [ ]:
v2.write.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_ofgenes_32_categories.parquet"
)

In [ ]:
v2.toPandas().to_csv(
    "/Users/yt4/Projects/Gentropy-manuscript/data/list_of_genes_32_categories.csv",
    index=False,
)